# Paralelna cjevovodna mreža: predvidi → izračunaj → provjeri

Tri grane spajaju ista dva čvora. Nepoznati su zajednički gubitak energije \(H\) i tri protoka. Faktor trenja ovisi o Reynoldsovu broju, pa sustav nije samo linearna podjela ukupnog protoka.

## Predvidi

1. Koja će grana preuzeti najveći protok: najkraća, najšira ili najglađa?
2. Moraju li protoci biti jednaki zato što je pad energije jednak?
3. Koja dva neovisna reziduala treba pratiti: čvorni i gransko-energetski?

Model koristi Darcy–Weisbachovu jednadžbu, lokalne gubitke i eksplicitnu Swamee–Jainovu turbulentnu aproksimaciju faktora trenja. Ovaj pokus rješava samo mreže u kojima su sve grane turbulentne (Re > 4000); prijelazno područje ne interpolira. Sintetički nastavni ulazi nisu mjerenja niti zamjena za kalibraciju stvarnog sustava. Nakon pokusa slijede zasebni podatci Z3 i Z6 iz poglavlja 13.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
g, nu = 9.81, 1.0e-6

branches = [
    {"name": "A", "L": 95.0, "D": 0.090, "eps": 0.00005, "K": 2.0},
    {"name": "B", "L": 70.0, "D": 0.075, "eps": 0.00015, "K": 3.5},
    {"name": "C", "L": 130.0, "D": 0.105, "eps": 0.00010, "K": 1.5},
]

def friction_factor(Re, rel_roughness):
    if Re <= 4000 or rel_roughness < 0:
        raise ValueError("Ovaj mrežni pokus traži Re > 4000 i ε/D ≥ 0; prijelaz se ne interpolira.")
    return 0.25/np.log10(rel_roughness/3.7 + 5.74/Re**0.9)**2

def minimum_turbulent_flow(branch):
    return 4001*nu*np.pi*branch["D"]/4

def branch_loss(Q, branch):
    if Q <= 0:
        return 0.0
    area = np.pi*branch["D"]**2/4
    velocity = Q/area
    Re = velocity*branch["D"]/nu
    lam = friction_factor(Re, branch["eps"]/branch["D"])
    return (lam*branch["L"]/branch["D"] + branch["K"])*velocity**2/(2*g)

def flow_for_head(head, branch, q_upper, iterations=70):
    lo, hi = minimum_turbulent_flow(branch), q_upper
    if head < branch_loss(lo, branch)-1e-12:
        raise ValueError("Zadana visina nema obuhvaćeno turbulentno rješenje.")
    while branch_loss(hi, branch) < head:
        hi *= 2
    for _ in range(iterations):
        mid = 0.5*(lo+hi)
        if branch_loss(mid, branch) < head:
            lo = mid
        else:
            hi = mid
    return 0.5*(lo+hi)

def solve_network(Q_total, branch_data, tolerance=1e-11, max_iterations=100):
    low = max(branch_loss(minimum_turbulent_flow(b), b) for b in branch_data)
    if sum(flow_for_head(low, b, Q_total) for b in branch_data) > Q_total:
        raise ValueError("Traženi protok nije obuhvaćen modelom sa svim turbulentnim granama.")
    high = max(branch_loss(Q_total, b) for b in branch_data)
    history = []
    for iteration in range(max_iterations):
        head = 0.5*(low+high)
        flows = np.array([flow_for_head(head, b, Q_total) for b in branch_data])
        residual = flows.sum()-Q_total
        history.append((iteration, head, residual))
        if abs(residual) < tolerance:
            return head, flows, np.asarray(history)
        if residual > 0:
            high = head
        else:
            low = head
    raise RuntimeError("Mrežni rješavač nije konvergirao.")

Q_total = 0.030
head, flows, history = solve_network(Q_total, branches)
losses = np.array([branch_loss(q, b) for q, b in zip(flows, branches)])
for b, q, h in zip(branches, flows, losses):
    print(f"Grana {b['name']}: Q={1e3*q:6.3f} L/s, h={h:.6f} m")
print(f"Čvorni rezidual = {flows.sum()-Q_total:.3e} m³/s; zajednički H = {head:.6f} m")

## Izračunaj: nelinearno rješenje i osjetljivost

Vanjska bisekcija mijenja zajednički \(H\) dok zbroj granskih protoka ne zatvori čvor. Unutarnja bisekcija za svaku granu invertira nelinearnu funkciju \(h_i(Q_i)\). Povijest vanjskog reziduala pokazuje stvarnu konvergenciju, a ne samo konačan broj.

Zatim mijenjamo promjer grane B. Za svaki novi promjer ponovno rješavamo cijelu mrežu; ostali protoci se također moraju prilagoditi.


In [ ]:
D2_values = np.linspace(0.065, 0.095, 31)
flow_sensitivity = []
for diameter in D2_values:
    modified = [dict(b) for b in branches]
    modified[1]["D"] = diameter
    _, q_modified, _ = solve_network(Q_total, modified)
    flow_sensitivity.append(q_modified)
flow_sensitivity = np.asarray(flow_sensitivity)

print(f"Promjena Q_B: {1e3*flow_sensitivity[0,1]:.3f} → {1e3*flow_sensitivity[-1,1]:.3f} L/s")
print(f"Broj vanjskih iteracija osnovnog slučaja: {len(history)}")


## Provjeri

Čvorna bilanca provjerava očuvanje mase. Raspon granskih gubitaka provjerava energijsku kompatibilnost. Monotonost \(Q_B(D_B)\) dodatna je fizikalna provjera osjetljivosti.


In [ ]:
reynolds_branches = np.array([4*q/(np.pi*b["D"]*nu) for q,b in zip(flows,branches)])
assert np.all(reynolds_branches>4000)
assert abs(flows.sum()-Q_total) < 1e-10
assert np.ptp(losses) < 1e-10
assert abs(history[-1,2]) < 1e-10
assert np.all(np.diff(flow_sensitivity[:,1]) > 0)
assert np.allclose(flow_sensitivity.sum(axis=1), Q_total, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="vanjska iteracija", ylabel=r"$|\sum Q_i-Q|$ (m³/s)", title="Rezidual nelinearne mreže")
for i, b in enumerate(branches):
    axes[1].plot(1e3*D2_values, 1e3*flow_sensitivity[:,i], label=f"Q_{b['name']}")
axes[1].set(xlabel="promjer grane B (mm)", ylabel="granski protok (L/s)", title="Preraspodjela cijele mreže")
axes[1].legend()
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()

## Z3: uravnoteženje dviju paralelnih grana

Ovo je **zaseban dvogranski model iz zadatka**, s već zadanim konstantnim otporima, a ne prethodna trogranska geometrija. $R_1=12000$ s²/m⁵, $R_2=48000$ s²/m⁵ i $Q=0{,}020$ m³/s. Regulacija održava ukupni protok i osigurava potrebnu visinu. Cilj je jednak protok u oba ogranka dodavanjem pasivnog ventila u samo jednu granu. Predvidi gdje ventil treba biti prije računa.

In [ ]:
R3 = np.array([12000.,48000.])
Q3 = 0.020
q3 = np.full(2,Q3/2)
valve_branch3 = int(np.argmin(R3))
Rv3 = np.max(R3)-np.min(R3)
effective_R3 = R3.copy();effective_R3[valve_branch3] += Rv3
losses3 = effective_R3*q3**2
valve_loss3 = Rv3*q3[valve_branch3]**2
print(f"Ventil u grani {valve_branch3+1}; Rv = {Rv3:.0f} s²/m⁵")
print(f"Protoci = {1000*q3} L/s; zajednički gubitci = {losses3} m; ventil = {valve_loss3:.2f} m")
assert valve_branch3==0 and Rv3>=0
assert np.isclose(q3.sum(),0.020,atol=1e-14)
assert np.allclose(losses3,[4.8,4.8],atol=1e-12)
assert np.isclose(valve_loss3,3.6,atol=1e-12)
assert R3[0]-R3[1]<0  # Sam ventil u grani 2 tražio bi negativan dodatni otpor.

## Z6: energija regulacije i granica zaključka o usisu

Koristimo točno sintetičke krivulje iz zadatka: $H_p=24-0{,}012q^2$, $H_s=5+0{,}025q^2$, $H_{s,V}=5+0{,}040q^2$, s $q$ u L/s i visinama u metrima. U oba načina rada pretpostavljeni su $ρ=1000$ kg/m³, ukupna učinkovitost 0,72 i 5000 h/god. NPSH krivulja $2+0{,}003q^2$ pripada **nazivnoj brzini**, pa iz nje ne izvodimo NPSH za sniženu brzinu.

In [ ]:
q6 = np.sqrt((24-5)/(0.012+0.040))
Q6 = q6*1e-3
HV6 = 24-0.012*q6**2
Hs6 = 5+0.025*q6**2
s6 = np.sqrt((Hs6+0.012*q6**2)/24)
rho6, eta6, hours6 = 1000.,0.72,5000.
PV6 = rho6*g*Q6*HV6/eta6
Ps6 = rho6*g*Q6*Hs6/eta6
EV6, Es6 = PV6*hours6/1e6, Ps6*hours6/1e6
NPSHa6 = 10.2-2.0-1.2-0.35
NPSHr6 = 2+0.003*q6**2  # samo pri nazivnoj brzini
print(f"q = {q6:.6f} L/s; HV = {HV6:.6f} m; Hs = {Hs6:.6f} m; s = {s6:.6f}")
print(f"EV = {EV6:.6f} MWh/god; Es = {Es6:.6f} MWh/god; ušteda = {EV6-Es6:.6f} MWh/god")
print(f"Nazivna brzina: NPSHa = {NPSHa6:.3f} m; NPSHr = {NPSHr6:.6f} m; razlika = {NPSHa6-NPSHr6:.6f} m")
assert abs(EV6-25.54)<0.005 and abs(Es6-18.41)<0.005
assert abs((EV6-Es6)-7.14)<0.005
assert np.isclose(HV6,5+0.040*q6**2,atol=1e-12)
assert np.isclose(24*s6**2-0.012*q6**2,Hs6,atol=1e-12)
assert 0<s6<1 and EV6>Es6
# Provjera energetskog lanca: električna ušteda odgovara uklonjenoj
# hidrauličkoj disipaciji tek nakon pretvorbe učinkovitosti.
assert np.isclose((PV6-Ps6)*eta6,rho6*g*Q6*(HV6-Hs6),rtol=1e-13)
assert abs(NPSHa6-NPSHr6-3.55)<0.005
fig,ax=plt.subplots(figsize=(5.8,3.6))
ax.bar(["prigušenje ventilom","regulacija brzinom"],[EV6,Es6],color=["#1565c0","#1e8449"])
ax.set(ylabel="električna energija (MWh/god)",title="Z6: isti protok, zadana stalna učinkovitost")
ax.grid(axis="y",ls=":",alpha=.4)
plt.tight_layout();plt.show()

## Protumači

1. Zašto povećanje promjera grane B u trogranskom pokusu mijenja i protoke ostalih grana? Što bi se izgubilo kad bi se promijenila samo brzina u grani B?
2. Promijeni lokalni koeficijent grane B za ±10 %. Kako se mijenja podjela protoka? Zašto taj zadani raspon nije automatski standardna mjerna nesigurnost?
3. Zašto ventil iz Z3 pripada grani manjeg otpora? Što bi se dogodilo s ukupnim protokom kada crpka ne bi mogla osigurati veću potrebnu visinu?
4. Koji dio razlike električnih snaga iz Z6 predstavlja uklonjena ventilska disipacija? Zašto nije ispravno izravno poistovjetiti električnu i hidrauličku snagu?
5. Zašto se rezultat NPSH-a pri nazivnoj brzini ne smije proglasiti jamstvom prihvatljivosti pri sniženoj? Koji podatci i kriterij margine nedostaju?
6. Koja se od pretpostavki stalne učinkovitosti, konstantnih otpora ili turbulentnog režima mora ponovno provjeriti pri većoj promjeni radne točke?